> ## SOLUTION / ANSWER KEY &mdash; Lab 10.9
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-09-responsible-checklist.ipynb`](../lab-09-responsible-checklist.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 10.9 &mdash; The Responsible-Agent Checklist

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 5 &middot; Module 10 &mdash; Ethics &amp; Responsible AI in Agentic Systems**

### What you'll do
- Turn the responsible-agent checklist into code
- Check grounding, least-privilege, HITL, observability, evals
- Gate deployment on every item passing

> **How this lab works (near-real):** read the **Concept**, fill the real `___` blanks in **Build it**, then **run it &amp; observe**. The responsible-AI logic here &mdash; injection defence, least privilege, trace-reading, fairness, the eval loop, the guardrails &mdash; is **real, deterministic Python** you can run offline. The **Advanced agent labs (10&ndash;12)** additionally drive a **real Groq model** through `create_agent`: **Run it for real** and **read the trace**. Finish with an open **Your turn**. There is **no auto-grader**; the goal is a responsible agent and a trace you can read.

> **Framework note:** these labs use the **real** LangChain 1.x (`langchain`, `langchain-core`, `langchain-groq`). The agent labs use a **real hosted model** (`ChatGroq("openai/gpt-oss-20b")` &mdash; reliable tool-calling via `create_agent`); the guardrail/eval/trace logic is genuine rule-based Python. If `GROQ_API_KEY` is unset, the run cells print how to set it instead of crashing. A `@tool` must **catch its own errors and return a string** &mdash; a tool that *raises* can abort the whole agent run. This is the **course finale** &mdash; Lab 5.2: responsible-AI frameworks &amp; **debugging agent errors**.

**Reference:** [Module 10 slides &mdash; The responsible-agent checklist](../../presentation/day5-module10-ethics-responsible-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 10 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, time, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY (+ other keys)

WORK = os.path.join(os.environ.get("TEMP") or os.environ.get("TMP") or "/tmp", "biaa-lab-10-09")
os.makedirs(WORK, exist_ok=True)

def groq_ready():
    """True if a GROQ_API_KEY is configured -- the 'Run it for real' cells self-skip if not."""
    return bool(os.environ.get("GROQ_API_KEY"))

from langchain_groq import ChatGroq
# Day-5 provider: a REAL hosted model with reliable tool-calling via create_agent.
MODEL = "openai/gpt-oss-20b"
llm = ChatGroq(model=MODEL, temperature=0) if groq_ready() else None

def with_backoff(fn, tries=5):
    """Run fn(); retry with backoff on Groq rate limits (HTTP 429). Other errors propagate."""
    last = None
    for attempt in range(tries):
        try:
            return fn()
        except Exception as e:
            last = e
            if "429" in str(e) or "rate limit" in str(e).lower() or "rate_limit" in str(e).lower():
                wait = 5 * (attempt + 1)
                print(f"(rate-limited -- retrying in {wait}s)")
                time.sleep(wait)
                continue
            raise
    raise last

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if groq_ready():
    print("Groq ready | model:", MODEL, "| WORK:", WORK)
else:
    print("GROQ_API_KEY not set -- add it to the repo .env (free key at console.groq.com).")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
Before an agent ships, you should answer **yes** to every item on the responsible-agent checklist (deck
slide 11): grounded, least-privilege, human-in-the-loop, observable, evaluated. Encode it as **automated
checks** over the agent's config and make deployment a **gate** &mdash; no item skipped, no exceptions.
The agent you assemble in Lab 11 is built to pass exactly this checklist.

In [ ]:
DANGEROUS = {"place_trade", "delete_records", "wire_funds"}
print("a config to gate:", {"grounds_and_cites": True, "tools": ["extract"], "human_approval": True, "traced": True, "eval_cases": 12})

## Build it
Complete `checklist` (per-item pass) and `ready_to_deploy` (all must pass).

In [ ]:
DANGEROUS = {"place_trade", "delete_records", "wire_funds"}

def checklist(cfg):
    return {
        "grounded": cfg.get("grounds_and_cites", False),
        "least_privilege": not any(t in DANGEROUS for t in cfg.get("tools", [])),
        "human_in_loop": cfg.get("human_approval", False),
        "observable": cfg.get("traced", False),
        "evaluated": cfg.get("eval_cases", 0) > 0,
    }

def ready_to_deploy(cfg):
    return all(checklist(cfg).values())

GOOD = {'grounds_and_cites': True, 'tools': ['extract', 'compute'], 'human_approval': True, 'traced': True, 'eval_cases': 12}

try:
    print("good config:", ready_to_deploy(GOOD))
    print("checklist  :", checklist(GOOD))
    print("dangerous  :", ready_to_deploy({**GOOD, "tools": ["place_trade"]}))
    print("no evals   :", ready_to_deploy({**GOOD, "eval_cases": 0}))
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- The `GOOD` config passes every item; swap in a dangerous tool and **least_privilege** fails the gate.
- Drop `eval_cases` to 0 and **evaluated** fails &mdash; an agent nobody measured doesn't ship.
- `ready_to_deploy` is one `all(...)`: governance you can run in CI, not a document nobody reads.

## Your turn (open task &mdash; no grader)
Add a sixth checklist item that matters to *you* &mdash; e.g. `"rate_limited"` (the agent wraps calls in
`with_backoff`) or `"input_sanitised"`. **What good looks like:** `ready_to_deploy` now also blocks on your
item, and you can defend why it belongs on the gate for a production agent.

In [ ]:
# --- Reference answer (ONE good way to do the 'Your turn' task -- compare with your own) ---
def checklist(cfg):
    return {
        "grounded": cfg.get("grounds_and_cites", False),
        "least_privilege": not any(t in DANGEROUS for t in cfg.get("tools", [])),
        "human_in_loop": cfg.get("human_approval", False),
        "observable": cfg.get("traced", False),
        "evaluated": cfg.get("eval_cases", 0) > 0,
        "rate_limited": cfg.get("uses_backoff", False),   # NEW: wraps provider calls in with_backoff
    }
def ready_to_deploy(cfg):
    return all(checklist(cfg).values())
print("GOOD (no backoff) :", ready_to_deploy(GOOD))                       # now blocked by the new gate
print("GOOD + backoff    :", ready_to_deploy({**GOOD, "uses_backoff": True}))

---
### Key takeaway
The checklist as a deployment gate makes responsibility non-optional: no agent ships unless it's grounded, least-privilege, human-gated, observable and evaluated. Governance you can run in CI, not a document nobody reads.

[&#8592; All Module 10 labs](./index.html) &nbsp;&middot;&nbsp; [Module 10 slides](../../presentation/day5-module10-ethics-responsible-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>